# Phase 1 — MedQA Uncertainty Experiment

Two-turn active-clarification pipeline on the MedQA held-out set.

**Pipeline per record:**
1. Model sees `ehr_summary` + `question` (no answer options) → generates clarifying question + preliminary assessment + confidence
2. Patient simulator answers the CQ from the combined `patient_context` / `nurse_context` / `specialist_context`
3. Model sees presentation + question + clarifying exchange + answer options → updated assessment + confidence

**Key outputs:** preliminary/updated correctness, confidence delta, clarifying question (classified by judge in Phase 2)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../../").resolve()))

# ── Dataset config ────────────────────────────────────────────────────────
DATASET          = "medqa"
ROOT             = Path("../../").resolve()
PROMPTS_DIR      = ROOT / "prompts"  / DATASET
DATASETS_DIR     = ROOT / "datasets" / DATASET
OUTPUTS_DIR      = ROOT / "outputs"  / DATASET

CASES_PATH       = DATASETS_DIR / "unseen_100.jsonl"   # held-out eval set (100 cases)
INSTRUCTION_FILE = PROMPTS_DIR  / "phase1_instruction.txt"
OUTPUT_CSV       = OUTPUTS_DIR  / "phase1_results.csv"

# ── Run config ────────────────────────────────────────────────────────────
N_RECORDS        = 100     # use all 100 held-out cases
MODEL_ID         = "gemini-2.5-flash"
REQUEST_INTERVAL = 3.0
RANDOM_SEED      = 42
# ─────────────────────────────────────────────────────────────────────────

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"ROOT:        {ROOT}")
print(f"Cases:       {CASES_PATH}")
print(f"Instruction: {INSTRUCTION_FILE}")
print(f"Output CSV:  {OUTPUT_CSV}")

In [ ]:
import importlib
import json
import logging
import os

import pandas as pd
from IPython.display import display, Markdown

import src.utils as utils_mod
import src.providers as providers_mod
import src.pipeline as pipeline_mod
importlib.reload(utils_mod)
importlib.reload(providers_mod)
importlib.reload(pipeline_mod)

from src.utils import load_dotenv, parse_json_response, format_answer_choices
from src.providers import GeminiProvider
from src.pipeline import Phase1Pipeline, PatientSimulator, TURN_0_SCHEMA, TURN_1_SCHEMA
from src.utils import SafetyBlockError

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)

load_dotenv(ROOT / ".env")
print("Environment loaded. src modules reloaded.")

## Smoke Test

In [ ]:
from google import genai
from google.genai import types

api_key = os.environ["VERTEX_API_KEY"]
client = genai.Client(api_key=api_key, http_options=types.HttpOptions(api_version="v1beta"))
resp = client.models.generate_content(
    model=MODEL_ID,
    contents="Reply with exactly: SMOKE TEST PASSED",
    config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=20),
)
print(f"Smoke test: {resp.text.strip()}")

## Load Dataset

In [ ]:
with open(CASES_PATH, encoding="utf-8") as f:
    raw_cases = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(raw_cases)} cases from {CASES_PATH.name}")

# Build records in pipeline format.
# Simulator context = patient + nurse + specialist (excludes full_patient_context
# which contains an explicit teaching point that reveals the answer).
records = []
for c in raw_cases:
    simulator_context = "\n\n---\n\n".join(
        ctx for ctx in [
            c.get("patient_context", ""),
            c.get("nurse_context", ""),
            c.get("specialist_context", ""),
        ] if ctx and ctx.strip()
    )
    records.append({
        "id":               c["case_id"],
        "ehr_summary":      c["ehr_summary"],
        "question":         c["question"],
        "options":          c["options"],
        "correct_option":   c["correct_option"],
        "correct_answer":   c["correct_answer"],
        "simulator_context": simulator_context,
        "difficulty":       c.get("difficulty", ""),
    })

print(f"Records prepared: {len(records)}")
print(f"\nDifficulty distribution:")
diff_counts = pd.Series([r['difficulty'] for r in records]).value_counts()
print(diff_counts.to_string())

print("\n--- Sample record ---")
r = records[0]
print(f"ID:            {r['id']} | difficulty: {r['difficulty']}")
print(f"EHR summary:   {r['ehr_summary']}")
print(f"Question:      {r['question']}")
print(f"Options:       {r['options']}")
print(f"Correct:       {r['correct_option']} | {r['correct_answer']}")
print(f"Simulator ctx: {r['simulator_context'][:200]}...")

## Preview Instruction

In [ ]:
print(INSTRUCTION_FILE.read_text(encoding="utf-8"))

## Dry Run — Single Record
Verify the full two-turn flow on one record before running everything.

In [ ]:
provider  = GeminiProvider(model_id=MODEL_ID)
simulator = PatientSimulator(provider)
instruction = INSTRUCTION_FILE.read_text(encoding="utf-8").strip()

test = records[1]   # pick second record for variety
print(f"Testing on: {test['id']} | difficulty={test['difficulty']}")
print(f"EHR:  {test['ehr_summary']}")
print(f"Q:    {test['question']}")
print()

# Turn 0
user_msg_0 = (
    f"Patient presentation:\n{test['ehr_summary'].strip()}\n\n"
    f"Clinical question:\n{test['question'].strip()}"
)
raw_0 = provider.call(
    system_instruction=instruction,
    user_message=user_msg_0,
    temperature=0.0,
    max_tokens=1024,
    expect_json=TURN_0_SCHEMA,
)
parsed_0 = parse_json_response(raw_0)
print("=== TURN 0 ===")
print(json.dumps(parsed_0, indent=2))

# Patient simulation
cq = parsed_0["clarifying_question"]
patient_answer = simulator.answer(cq, test["simulator_context"])
print(f"\n=== PATIENT RESPONSE ===\n{patient_answer}")

# Turn 1
from src.pipeline import _POST_CLARIFICATION_INSTRUCTION
user_msg_1 = (
    f"Patient presentation:\n{test['ehr_summary'].strip()}\n\n"
    f"Your clarifying question:\n{cq}\n\n"
    f"Patient's answer:\n{patient_answer}\n\n"
    f"Clinical question:\n{test['question'].strip()}\n\n"
    f"Answer choices:\n{format_answer_choices(test['options'])}"
)
raw_1 = provider.call(
    system_instruction=_POST_CLARIFICATION_INSTRUCTION,
    user_message=user_msg_1,
    temperature=0.0,
    max_tokens=512,
    expect_json=TURN_1_SCHEMA,
)
parsed_1 = parse_json_response(raw_1)
print(f"\n=== TURN 1 ===\n{json.dumps(parsed_1, indent=2)}")
print(f"\nCorrect answer: {test['correct_option']} | {test['correct_answer']}")

## Run Full Experiment
Processes all 100 held-out cases. Resumes automatically if interrupted.

In [ ]:
provider = GeminiProvider(model_id=MODEL_ID)

pipeline = Phase1Pipeline(
    provider=provider,
    instruction_file=INSTRUCTION_FILE,
    output_csv=OUTPUT_CSV,
    request_interval=REQUEST_INTERVAL,
)

pipeline.run(records)

## Results Summary

In [ ]:
results_df = pd.read_csv(OUTPUT_CSV)
valid = results_df[~results_df["was_blocked"]]

print(f"Total records:           {len(results_df)}")
print(f"Blocked:                 {results_df['was_blocked'].sum()}")
print(f"Valid:                   {len(valid)}")
print()
print(f"Preliminary correct:     {valid['is_correct_preliminary'].sum()} / {len(valid)} "
      f"({valid['is_correct_preliminary'].mean():.1%})")
print(f"Updated correct:         {valid['is_correct_updated'].sum()} / {len(valid)} "
      f"({valid['is_correct_updated'].mean():.1%})")
print()
print(f"Mean prelim confidence:  {valid['preliminary_confidence'].mean():.1f}")
print(f"Mean updated confidence: {valid['updated_confidence'].mean():.1f}")
print(f"Mean confidence delta:   {valid['confidence_delta'].mean():.1f}")
print()

print("Confidence delta distribution:")
print(f"  Increased: {(valid['confidence_delta'] > 0).sum()}")
print(f"  No change: {(valid['confidence_delta'] == 0).sum()}")
print(f"  Decreased: {(valid['confidence_delta'] < 0).sum()}")
print()

print("Per-difficulty breakdown:")
display(valid.groupby('difficulty')[['is_correct_preliminary','is_correct_updated','confidence_delta']]
        .agg({'is_correct_preliminary':'mean','is_correct_updated':'mean','confidence_delta':'mean'})
        .round(3))

In [ ]:
# Full per-record detail for qualitative inspection
display(Markdown("**Per-record results (first 10):**"))
for _, row in results_df.head(10).iterrows():
    print("="*80)
    print(f"ID:          {row['id']} | difficulty={row['difficulty']}")
    print(f"EHR:         {row['ehr_summary']}")
    print(f"CQ:          {row['clarifying_question']}")
    print(f"Patient:     {row['patient_response']}")
    print(f"Prelim:      {row['preliminary_assessment']} (conf={row['preliminary_confidence']})")
    print(f"Updated:     {row['updated_assessment']} (conf={row['updated_confidence']})")
    print(f"Correct:     {row['correct_option']} | {row['correct_answer']}")
    print(f"Prelim OK:   {row['is_correct_preliminary']}  |  Updated OK: {row['is_correct_updated']}  |  Δconf: {row['confidence_delta']:+}")
    print()